In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameters.
# NOTE: the prior bowaka_v2_walkforward_optuna.yml is QUARANTINED (realism
# audit 2026-05-22 §P0-001). The default below points at the Phase-8
# walk-forward-purpose contract-parity config that matches today's lake
# (IEX-only, current_code_parity). FEED='auto' upgrades simulation.mode
# to intended_realism the moment SIP bars+quotes land in the lake.
# Walk-forward sizing is locked to the operator spec (2026-05-23):
# train=21, val=1, final_holdout=5 -> 3 folds over the ~29.8-month IEX lake.
#
# Speedup report v2 §4 P2 / Phase 2 — default points at the workstation
# overlay (192 GiB / 18-core box; reserve 62 GiB; strict_parallel=true;
# max_workers=8). The base optuna config carries the more conservative
# reserve_system_gib=32 / strict_parallel=false defaults; flip
# CONFIG_PATH below if you intentionally want those.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_actual_iex_current_code_optuna.workstation.yml'
N_TRIALS = None          # None -> optuna.n_trials from the config; or set an integer
N_STARTUP_TRIALS = None  # None -> optuna.n_startup_trials; random trials before TPE
N_JOBS = None            # None -> optuna.n_jobs (Phase 5 default: 8 workers on PostgreSQL)
FEED = 'auto'            # 'auto' (SIP > IEX > synthetic) | 'sip' | 'iex' | 'synthetic'
# Phase 10 default-on: trial 0 is pinned to the actual-contract
# parameter set (the live config) so the optimizer's best can be
# compared against the live incumbent. Set False to disable.
INCUMBENT_TRIAL = True


# 10 - Walk-Forward Optuna

Runs a **real** walk-forward parameter optimization against the shared
market-data lake. Each Optuna trial samples a parameter set, applies it
to the config, and runs a real backtest over every walk-forward
validation window; the trial objective is the median fold score. The
final-holdout window is never read during tuning.

**Feed auto-selection (`FEED='auto'`):** the notebook probes the lake
and adapts the run to the best data available -

| Lake holds | feed | simulation.mode |
|---|---|---|
| SIP bars + SIP quotes | `sip` | `intended_realism` |
| SIP bars, no SIP quotes | `sip` | `current_code_parity` |
| IEX bars only | `iex` | `current_code_parity` |
| neither | - | `smoke_fixture` (synthetic) |

Set `FEED` to `sip` / `iex` / `synthetic` to override the probe.

**Parameters:** `N_TRIALS` is the total trial count (`None` -> the
config's `optuna.n_trials`). `N_STARTUP_TRIALS` is how many of those are
random-sampling trials before TPE-guided search begins.

**Compute:** a run is `N_TRIALS` x `n_folds` real backtests - the config
default is a multi-day job. Set a small `N_TRIALS` for a quick run.

In [3]:
# Probe the lake and adapt the config's feed + simulation.mode.
from bowaka_v2_lab.optuna.autoconfig import resolve_walkforward_config
resolved = resolve_walkforward_config(CONFIG_PATH, feed_override=FEED)
print(f'feed   : {resolved.feed}')
print(f'mode   : {resolved.mode}')
print(f'reason : {resolved.reason}')
print(f'config : {resolved.path}')


feed   : iex
mode   : current_code_parity
reason : no SIP data; IEX bars present — current_code_parity on IEX (reproduces the live IEX paper strategy; IEX is partial-tape)
config : research_notebooks/bowaka_v2_lab/artifacts/resolved_configs/walkforward_resolved.yml


In [4]:
import json
from bowaka_v2_lab.optuna.walkforward_runner import run_walkforward_study
# Realism remediation 2 Phase 8 (audit §P0-011): current_code_parity
# studies require an explicit opt-in. ResolvedWalkforwardConfig
# carries the auto-opt-in flags when the lake forces parity mode
# (IEX-only / SIP-bars-no-quotes); the mechanical cap is research_only.
result = run_walkforward_study(
  resolved.path, n_trials=N_TRIALS, n_jobs=N_JOBS,
  n_startup_trials=N_STARTUP_TRIALS, allow_smoke=resolved.allow_smoke,
  allow_current_code_parity_study=resolved.allow_current_code_parity_study,
  tier=resolved.tier,
  incumbent_trial=INCUMBENT_TRIAL,
)
print(json.dumps(result, indent=2, default=str))


2026-05-30 12:09:42,750 INFO preflight passed: 4 checks
2026-05-30 12:55:59,663 INFO full per-fold preflight passed (mode=current_code_parity): 4 folds, 16 checks, 2 partial-tape limitation(s): ['missing_historical_quotes', 'missing_sip']
/quants-lab/research_notebooks/bowaka_v2_lab/src/bowaka_v2_lab/optuna/dispatcher.py:78: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
[I 2026-05-30 12:56:00,005] A new study created in RDB with name: iex__bowaka_v2_iex_walkforward_conservative_b44ea02b_20260530
2026-05-30 12:56:00,136 INFO walk-forward study iex__bowaka_v2_iex_walkforward_conservative_b44ea02b_20260530: 200 trials (25 random startup) x 3 folds, feed=iex, search_space_version=3; per-fold universe = daily point-in-time set (~697 eligible on 2025-08-27) (preflight probe sampled 100 symbols)
2026-05-30 12:56:00,177 INFO incumbent baseline: enqueued trial 0 with 30 params (all from th

KeyboardInterrupt: 